# 12 · Architectures & Training — Beating Spectral Bias

**Level:** Research
**Prerequisites:** Beginner–Advanced track

---

## 🧭 The Villain: Spectral Bias

Throughout this series, one problem kept appearing but was never fully addressed: PINNs struggle with **high-frequency solutions**. A network trained on the 1-D wave equation with $u(x,0)=\sin(4\pi x)$ fits worse than one with $u(x,0)=\sin(\pi x)$. This is not an accident — it is a fundamental property of neural networks called **spectral bias**.

### What is spectral bias?

Rahaman et al. (2019) and Tancik et al. (2020) proved and demonstrated experimentally that:

> **Standard MLPs learn low-frequency components of a function before high-frequency ones, regardless of how training data is presented.**

In the Fourier domain, a function $f(x)$ has components at many frequencies $\omega$. An MLP trained with gradient descent:
1. First fits the $\omega = 0$ component (constant mean) — epoch ~100
2. Then fits $\omega = 1$ (slow variation) — epoch ~1000  
3. Then fits $\omega = 2$ — epoch ~10,000
4. High frequencies $\omega \gg 1$ may **never** be learned in a practical training budget

This is because the Neural Tangent Kernel (NTK) of an MLP decays with frequency — the gradient signal for high-frequency components is exponentially weaker than for low-frequency ones.

### Why PINNs suffer more than standard ML

For a standard regression problem, you can always add more data near the difficult high-frequency features. For a PDE:
- The "training targets" are the physics residuals, not explicit function values
- The physics constraint couples all frequencies together
- A network that cannot represent the high-frequency component cannot satisfy the PDE at all, causing the loss to plateau

This is why the Burgers equation near shock formation is hard, why the wave equation with high wave numbers fails, and why stiff ODEs with fast oscillations defeat plain PINNs.

### What you will learn
1. A reproducible demonstration of spectral bias on a Poisson equation with high-frequency solution
2. Random Fourier feature embedding: the principal fix for spectral bias
3. The mathematical mechanism behind why Fourier features work
4. A comprehensive map of the modern PINN training toolkit

### References
- Rahaman et al. (2019), *On the spectral bias of neural networks*, ICML
- Tancik et al. (2020), *Fourier features let networks learn high-frequency functions*, NeurIPS
- Wang, Wang & Perdikaris (2021), *On the eigenvector bias of Fourier feature networks*, CMAME
- Wang, Sankaran, Wang & Perdikaris (2023), *An expert's guide to training PINNs*, CMAME
- Wang et al. (2024), *PirateNets: Physics-informed deep learning with residual adaptive networks*

## 1 · A deliberately hard target — the high-frequency Poisson equation

### The equation

1-D Poisson equation:
$$u''(x) = f(x), \qquad x\in[0,1], \qquad u(0)=u(1)=0$$

We choose $f$ so the exact solution has a high-frequency component:
$$u^*(x) = \sin(6\pi x) \quad \Rightarrow \quad f(x) = -(6\pi)^2\sin(6\pi x) \approx -1422 \sin(6\pi x)$$

### Why $k = 6\pi$ is hard

The target has **6 full oscillations** on $[0,1]$. The largest Fourier coefficient is at frequency $k = 6\pi \approx 18.85$. For a tanh-MLP with Xavier initialisation, the NTK eigenvalue at frequency $\omega$ decays roughly as $e^{-\omega^2}$ — so frequency $6\pi$ gets gradient signal $\sim e^{-355}$ times weaker than frequency 0. The plain MLP will struggle severely.

### Why Poisson (not ODE)?

Second-order problems like Poisson require computing $u''$ (second derivative via autograd), which is slightly more demanding than $u'$. This is the simplest PDE that:
1. Is 2nd order (involves $u_{xx}$, common in physics)
2. Has Dirichlet BCs at both endpoints (zero at both ends)
3. Has no time dimension (pure spatial, easy to verify)

The exact solution is known analytically, so we can measure accuracy precisely.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

k = 6*np.pi
def exact(x): return np.sin(k*x)
def f_rhs(x): return -(k**2)*torch.sin(k*x)

xf = torch.linspace(0,1,400,device=device).view(-1,1).requires_grad_(True)
xb = torch.tensor([[0.0],[1.0]],device=device)


### 📌 Code setup notes

```python
k = 6 * np.pi                        # angular frequency of target solution
def f_rhs(x): return -(k**2) * sin(k*x)   # forcing term f(x) = -k²sin(kx)

xf = linspace(0, 1, 400).requires_grad_(True)   # 400 physics collocation points
xb = tensor([[0.0], [1.0]])                      # 2 boundary points
```

**Why 400 collocation points for a 1-D problem?**

The target has 6 full oscillations on $[0,1]$. By the Nyquist theorem, you need at least 2 points per oscillation cycle to even *represent* the target. With 6 oscillations: minimum ~12 points. But for the PDE residual (which involves $u''$), you need much denser sampling to accurately estimate second derivatives. 400 points gives ~67 points per oscillation — comfortably above Nyquist.

**The forcing term:**
$$f(x) = -(6\pi)^2 \sin(6\pi x) \approx -1422 \sin(6\pi x)$$

This large coefficient ($\approx 1422$) means the PINN loss has a large scale near the target — the residual $u'' - f$ will be numerically large during early training when $u''$ is near zero. This can cause the plain MLP's loss to plateau at a high value, a symptom of spectral bias.

In [ ]:
class PlainMLP(nn.Module):
    def __init__(self, w=64, d=4):
        super().__init__()
        layers=[1]+[w]*d+[1]; seq=[]
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i],layers[i+1]))
            if i<len(layers)-2: seq.append(nn.Tanh())
        self.net=nn.Sequential(*seq)
    def forward(self,x): return self.net(x)

class FourierMLP(nn.Module):
    # random Fourier features: x -> [cos(Bx), sin(Bx)] then MLP
    def __init__(self, w=64, d=4, n_feat=64, sigma=6.0):
        super().__init__()
        B = torch.randn(1, n_feat, device=device)*sigma
        self.register_buffer("B", B)
        layers=[2*n_feat]+[w]*d+[1]; seq=[]
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i],layers[i+1]))
            if i<len(layers)-2: seq.append(nn.Tanh())
        self.net=nn.Sequential(*seq)
    def features(self,x):
        proj = 2*np.pi*x @ self.B
        return torch.cat([torch.cos(proj), torch.sin(proj)],dim=1)
    def forward(self,x): return self.net(self.features(x))


### 📌 Architecture deep-dive — PlainMLP vs FourierMLP

#### PlainMLP — the baseline

```python
class PlainMLP(nn.Module):
    # 1 → 64 → 64 → 64 → 64 → 1  (4 hidden layers)
    self.net = Sequential(Linear(1,64), Tanh(), ..., Linear(64,1))
```

Standard architecture, identical to what we've used throughout the series. Its weakness: the initial linear projection `Linear(1, 64)` maps a scalar $x$ into 64 dimensions. The resulting features are **linear functions of $x$** before the first Tanh — so the very first feature layer can only express low-frequency structure. Deeper layers can compose these to get higher frequencies, but the gradient signal for high-frequency components remains weak throughout training.

#### FourierMLP — the fix

```python
class FourierMLP(nn.Module):
    def __init__(self, n_feat=64, sigma=6.0):
        # Fixed random frequency matrix: shape (1, 64)
        B = torch.randn(1, n_feat) * sigma
        self.register_buffer("B", B)   # not trained — fixed!

        # MLP takes the 128-dim Fourier embedding as input:
        self.net = Sequential(Linear(128, 64), Tanh(), ..., Linear(64, 1))

    def features(self, x):
        proj = 2 * pi * x @ B         # project x onto random frequencies
        return cat([cos(proj), sin(proj)], dim=1)   # Fourier features

    def forward(self, x):
        return self.net(self.features(x))
```

**Step 1 — Random Fourier projection:**
$$z = 2\pi\mathbf{B}x, \qquad \mathbf{B} \sim \mathcal{N}(0, \sigma^2)$$

Each row of $\mathbf{B}$ is a random frequency $\omega_k \sim \mathcal{N}(0, \sigma^2)$. The projection produces $n_\text{feat}$ random frequencies, broadly centred around scale $\sigma$.

**Step 2 — Sinusoidal features:**
$$\gamma(x) = [\cos(\omega_1 x), \sin(\omega_1 x), \cos(\omega_2 x), \sin(\omega_2 x), ..., \cos(\omega_{64} x), \sin(\omega_{64} x)]$$

This is a **random Fourier feature map** (Rahimi & Recht 2007). The output is always 128-dimensional for any input dimensionality.

**Why this defeats spectral bias:**

The NTK of a network with Fourier embedding has a very different spectrum. When the input to the first linear layer is $\gamma(x)$ (a mix of sinusoids at many frequencies), the network can represent any linear combination of those sinusoids from the first layer forward. The gradient signal for frequency $\omega_k$ no longer decays with $\omega_k$ — all represented frequencies get roughly equal gradient signal.

**The `sigma` hyperparameter:**
- `sigma` controls the *scale* of the random frequencies
- `sigma = 1.0`: mostly low frequencies — misses the $6\pi$ component
- `sigma = 6.0`: frequencies centred near $k = 6\pi$ — matches the target (this is why `sigma=6.0` is used)
- `sigma = 20.0`: too many very high frequencies — can produce noisy, hard-to-train features

**`register_buffer` — why B is not trained:**
```python
self.register_buffer("B", B)
```
$\mathbf{B}$ is drawn randomly once and **fixed** — it is not updated by the optimizer. If $\mathbf{B}$ were trainable, the network might collapse the frequencies during training, destroying the spectral coverage. Fixed random frequencies guarantee the feature map covers a range of scales throughout training. (Learnable versions exist — see SIREN, parametric Fourier features — but fixed random is simpler and often equally effective.)

In [ ]:
def train(model, epochs=8000, lr=1e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    for ep in range(epochs):
        opt.zero_grad()
        u=model(xf)
        u_x=torch.autograd.grad(u,xf,torch.ones_like(u),create_graph=True)[0]
        u_xx=torch.autograd.grad(u_x,xf,torch.ones_like(u_x),create_graph=True)[0]
        L=torch.mean((u_xx-f_rhs(xf))**2)+50*torch.mean(model(xb)**2)
        L.backward(); opt.step()
    return model

def evaluate(model):
    xs=np.linspace(0,1,400)
    xt=torch.tensor(xs,dtype=torch.float32,device=device).view(-1,1)
    with torch.no_grad(): up=model(xt).cpu().numpy().ravel()
    ue=exact(xs); return np.linalg.norm(up-ue)/np.linalg.norm(ue), xs, up, ue


In [ ]:
m_plain = train(PlainMLP().to(device))
e_plain,xs,up_plain,ue = evaluate(m_plain)
print(f"[plain MLP]     rel L2 = {e_plain:.3e}")

m_fourier = train(FourierMLP().to(device))
e_four,_,up_four,_ = evaluate(m_fourier)
print(f"[Fourier feats] rel L2 = {e_four:.3e}")


### 📌 Reading the results — what the numbers mean

The two models are compared by **relative L2 error**:
$$\text{rel. L2} = \frac{\|u_{\text{pred}} - u^*\|_2}{\|u^*\|_2}$$

**Typical results for this problem:**
- `[plain MLP] rel L2 ≈ 0.5–1.0` — the network learns essentially nothing about the high-frequency oscillations; it may produce a flat or slowly varying prediction
- `[Fourier features] rel L2 ≈ 0.001–0.01` — the network correctly captures most of the $\sin(6\pi x)$ shape

The 100–1000× accuracy gap for the **same number of parameters, same number of epochs, same architecture depth** makes the point powerfully.

**Why the plain MLP produces ~0.5 error:**
With rel L2 ≈ 0.5 on $\sin(6\pi x)$ (which has L2 norm = $1/\sqrt{2}$), the plain MLP is capturing barely more than the zero function. It has learned that the function averages to zero and has some small amplitude, but it cannot resolve the 6 oscillations.

**Practical guidance on `sigma`:**
```python
# If you know the target frequency k:
sigma ≈ k / (2*pi)   # center the random frequencies near the target
# For sin(6πx): sigma ≈ 6π/(2π) = 3 to 6  ✓
```
In practice, you rarely know the target frequency in advance (that's the point — the PDE determines it). A common approach: run with `sigma` values of `{1, 3, 6, 10}` and pick the one with the lowest validation error.

In [ ]:
plt.figure(figsize=(9,4))
plt.plot(xs,ue,'k-',lw=2,label='exact  sin(6πx)')
plt.plot(xs,up_plain,'b--',lw=2,label=f'plain MLP (L2={e_plain:.1e})')
plt.plot(xs,up_four,'r-.',lw=2,label=f'Fourier features (L2={e_four:.1e})')
plt.legend(); plt.xlabel('x'); plt.ylabel('u'); plt.title('Fourier features defeat spectral bias')
plt.tight_layout(); plt.show()


## 2 · The Modern PINN Training Toolkit — A Field Guide

Fourier features fix spectral bias. But the literature has identified many other failure modes and corresponding fixes. The table below summarises the current best practice (as of Wang et al. 2023, *An expert's guide to training PINNs*).

---

### Problem 1: Imbalanced loss terms → Self-adaptive weighting

**Symptom:** One loss component (e.g. physics) converges fast, another (e.g. BC) converges slow. The total loss decreases but the badly-balanced term stays large, causing poor accuracy for that constraint.

**Fix:** Rescale each loss term so their gradient magnitudes are roughly equal:
$$\lambda_k \leftarrow \frac{\max_j \|\nabla_\theta \mathcal{L}_j\|}{\|\nabla_\theta \mathcal{L}_k\|}$$

This NTK-based weighting (Wang, Yu & Perdikaris 2022) adjusts automatically every few hundred epochs. An easier version: just multiply the BC/IC loss by a large constant (e.g. 100) as a manual fix.

---

### Problem 2: Temporal causality violations → Causal training

**Symptom:** For time-dependent problems, the PINN learns the solution at large $t$ before learning $t=0$. This creates propagating errors because the IC is violated.

**Fix:** Weight the physics loss by $e^{-\epsilon\, t}$, so early times contribute more to the loss. Then slowly reduce $\epsilon$ (or switch to a curriculum: first train on $t\in[0,0.25]$, then $[0,0.5]$, etc.).

Wang, Sankaran & Perdikaris (2022) showed this can recover accuracy on stiff oscillators where plain PINNs catastrophically fail.

---

### Problem 3: Gradient pathologies in deep networks → Modified MLP

**Symptom:** Training a 6+ layer MLP causes vanishing/exploding gradients. Loss oscillates or stalls.

**Fix (Modified MLP, Wang, Teng & Perdikaris 2021):** Add two parallel "encoder" layers $U = \tanh(W_1 x + b_1)$, $V = \tanh(W_2 x + b_2)$ at the input. Before each hidden layer $H_k$, apply gating:
$$H_k \leftarrow (1 - H_{k-1}) \cdot U + H_{k-1} \cdot V$$

This creates a "highway" architecture that prevents gradient vanishing and allows very deep PINNs to train stably.

---

### Problem 4: Soft BC enforcement → Hard constraint ansatz

**Symptom:** BCs/ICs are satisfied to tolerance $\sim 10^{-3}$ but you need $\sim 10^{-6}$ (e.g. for conservation law problems).

**Fix:** Build the BC/IC satisfaction into the network architecture rather than penalising it:

For a 1-D problem with $u(0) = 0, u(1) = 0$:
$$\hat{u}(x) = x(1-x) \cdot \text{Network}(x)$$

$\hat{u}$ satisfies the BCs *exactly* for any network output, regardless of training. The network only needs to learn the interior behaviour. This removes the BC from the loss entirely, leaving only the physics residual.

---

### Problem 5: Large/multiscale domains → Domain decomposition

**Symptom:** Long-time or large-domain problems where the solution has very different behaviour in different regions. A single MLP with global representational capacity struggles.

**Fix:** Split the domain into smaller subdomains, train separate PINNs in each, and enforce continuity at the interfaces:
- **cPINN** (Conservative PINN): conservation form matching at interfaces
- **XPINN** (Extended PINN): general interface conditions, subdomains can overlap
- **FBPINN** (Finite Basis PINN): windowed basis functions, like FEM basis for PINNs

---

### Problem 6: Training very deep PINNs stably → PirateNets (2024)

**Symptom:** Deeper networks (>6 layers) have highly non-convex loss landscapes. Initialisation matters enormously; most runs fail.

**Fix (PirateNets, Wang et al. 2024):** Physics-informed initialisation + residual adaptive connections. The network is initialised so the physics residual is exactly satisfied by the initial (random) network, and residual connections allow gradients to flow to all layers without vanishing. State-of-the-art accuracy on many benchmark PDEs.

---

### Summary table — when to use each technique

| Technique | Use when | Complexity |
|---|---|---|
| **Adam → L-BFGS** | Final accuracy needed | Low — 2-line change |
| **Fourier features** | High-frequency solution | Low — wrapper class |
| **Modified MLP** | Deep network (>4 layers) | Medium |
| **Self-adaptive weights** | Loss components imbalanced | Medium |
| **Causal training** | Time-dependent, stiff | Medium |
| **Hard constraints** | Exact BC/IC required | Medium |
| **RAR/RAD sampling** | Sharp features, boundary layers | Medium (NB-08) |
| **Domain decomposition** | Large domain, multiscale | High |
| **PirateNets** | Very deep network, hard problem | High |
| **PIKANs** | Interpretability + accuracy | High (new) |

The good news: for most problems in this series, **Fourier features + self-adaptive weights + RAR sampling** solves the most common failure modes. Start there before reaching for more complex tools.